In [1]:
import os

import cv2
import torch
import torchvision.transforms.functional as F
from torchvision import transforms
from PIL import Image
import mss
import numpy as np
import math
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp

from detectors import DETECTOR

import yaml
import time

from collections import deque

import joblib
import numpy as np

In [2]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

models = []

config = load_config("config/detector/clip_base.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/clip.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

models.append(model)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done for clip!')

model.eval().to(device)

config = load_config("config/detector/i3d.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/i3d.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

models.append(model)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done I3D!')

model.eval().to(device)

config = load_config("config/detector/xception.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/xception.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done xception!')

model.eval().to(device)

models.append(model)

config = load_config("config/detector/spsl.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-fs-ff/spsl.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done SPSL!')

model.eval().to(device)

models.append(model)

cuda_streams = [torch.cuda.Stream() for _ in range(len(models))]


meta_model = joblib.load("fusion/regression_df40.pkl")

Appareil utilisé : cuda
===> Load checkpoint done for clip!
Appareil utilisé : cuda
64
loading pretrained model from /home/antoine/DF40/DeepfakeBench_DF40/training/pretrained/I3D_8x8_R50.pth
===> Load checkpoint done I3D!
Appareil utilisé : cuda
===> Load checkpoint done xception!
Appareil utilisé : cuda
===> Load checkpoint done SPSL!


In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

NORM_STATS = [{'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]}, {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}]

i3d_index = -1
for i, model in enumerate(models):
    if "I3D" in type(model).__name__:
        i3d_index = i
        break

SEQ_LEN = 16
noms_colonnes = ['clip_pred', 'i3d_pred', 'xception_pred', 'spsl_preds']

active_trackers = [] 

base_options = python.BaseOptions(model_asset_path='facedetection/blaze_face_short_range.tflite')
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
face_detector = vision.FaceDetector.create_from_options(options)

sct = mss.mss()
zone_capture = {"top": 1000, "left": 400, "width": 800, "height": 600}

active_trackers = [] 

base_options = python.BaseOptions(model_asset_path='facedetection/blaze_face_short_range.tflite')
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
face_detector = vision.FaceDetector.create_from_options(options)

sct = mss.mss()
zone_capture = {"top": 1000, "left": 400, "width": 800, "height": 600}

# preprocess = transforms.Compose([
#     transforms.Resize((224, 224)), 
#     transforms.ToTensor(),
# ])

print("Compiling")
for i in range(len(models)):

    models[i] = torch.compile(models[i])
print("Compiling done")

start_events = [torch.cuda.Event(enable_timing=True) for _ in range(len(models))]
end_events = [torch.cuda.Event(enable_timing=True) for _ in range(len(models))]
noms_modeles = ['CLIP', 'I3D', 'Xception', 'SPSL']

while True:
    start_time = time.perf_counter()
    
    sct_img = sct.grab(zone_capture)
    frame = cv2.cvtColor(np.array(sct_img), cv2.COLOR_BGRA2BGR)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    small_frame = cv2.resize(frame_rgb, (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=small_frame)
    results = face_detector.detect(mp_image)
    
    faces = []
    
    if results.detections:
        for detection in results.detections:
            bbox = detection.bounding_box

            x = int(bbox.origin_x * 2)
            y = int(bbox.origin_y * 2)
            w = int(bbox.width * 2)
            h = int(bbox.height * 2)
            
            if x < 0: 
                w += x
                x = 0
            if y < 0: 
                h += y
                y = 0
                
            faces.append((x, y, w, h))

    current_frame_trackers = []

    end_time = 0

    for face_i, (x, y, w, h) in enumerate(faces):
        cx = x + w // 2
        cy = y + h // 2
        
        best_tracker_idx = -1
        min_dist = float('inf')
        
        for i, tracker in enumerate(active_trackers):
            tcx, tcy = tracker['center']
            dist = math.hypot(cx - tcx, cy - tcy)
            if dist < min_dist and dist < min(w, h):
                min_dist = dist
                best_tracker_idx = i
                
        if best_tracker_idx != -1:
            matched_tracker = active_trackers.pop(best_tracker_idx)
            current_history = matched_tracker['history']
            current_frames = matched_tracker['frames']
        else:
            current_history = deque(maxlen=32)
            current_frames = deque(maxlen=SEQ_LEN)
            
        margin_x = int(w * 0.32)
        margin_y = int(h * 0.32)
        
        x1 = max(0, x - margin_x)
        y1 = max(0, y - margin_y)
        x2 = min(frame.shape[1], x + w + margin_x)
        y2 = min(frame.shape[0], y + h + margin_y)
        
        face_rgb_cropped = frame_rgb[y1:y2, x1:x2]
        pil_image = Image.fromarray(face_rgb_cropped)
        
        face_rgb_cropped = frame_rgb[y1:y2, x1:x2]

        face_resized = cv2.resize(face_rgb_cropped, (224, 224), interpolation=cv2.INTER_LINEAR)
        
        tensor_cpu = torch.from_numpy(face_resized).permute(2, 0, 1).float() / 255.0
        tensor_normalized = F.normalize(tensor_cpu, mean=NORM_STATS[0]["mean"], std=NORM_STATS[0]["std"])
        input_tensor = tensor_normalized.unsqueeze(0).pin_memory().to(device, non_blocking=True)
        
        tensor_norm_i3d = F.normalize(tensor_cpu, mean=NORM_STATS[1]["mean"], std=NORM_STATS[1]["std"])
        input_tensor_i3d = tensor_norm_i3d.unsqueeze(0).pin_memory().to(device, non_blocking=True)
        current_frames.append(input_tensor_i3d.squeeze(0))

        if len(current_frames) < SEQ_LEN:
            padded_frames = list(current_frames) + [current_frames[-1]] * (SEQ_LEN - len(current_frames))
        else:
            padded_frames = list(current_frames)

        input_tensor_3d = torch.stack(padded_frames, dim=0).unsqueeze(0)
        
        current_frame_trackers.append({'center': (cx, cy), 'history': current_history, 'frames': current_frames})

        raw_outputs = [None] * len(models)


        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                for i, (model, stream) in enumerate(zip(models, cuda_streams)):
                    with torch.cuda.stream(stream):
                        
                        if i == i3d_index:
                            data_dict = {'image': input_tensor_3d}
                        else:
                            data_dict = {'image': input_tensor}
                            
                        output = model(data_dict, inference=True)
                        raw_outputs[i] = output["prob"].clone() 
                        start_events[i].record(stream)
                        
                        output = model(data_dict, inference=True)
                        raw_outputs[i] = output["prob"].clone() 
                        
                        end_events[i].record(stream)
                    
        torch.cuda.synchronize()

        # for i in range(len(models)):
        #     # elapsed_time renvoie automatiquement le résultat en millisecondes
        #     temps_ms = start_events[i].elapsed_time(end_events[i])
        #     print(f"Model {noms_modeles[i]} : {temps_ms:.1f} ms")

        probabilities = [tensor.item() for tensor in raw_outputs]
        
        noms_colonnes = ['clip_pred', 'i3d_pred', 'xception_pred', 'spsl_preds']

        features_np = np.array(probabilities).reshape(1, -1)
        prob_ensemble = float(meta_model.predict_proba(features_np)[0][1])
        
        prob_ensemble = float(meta_model.predict_proba(features_np)[0][1])
        #prob_ensemble = float(meta_model.predict(features_df)[0])

        current_history.append(prob_ensemble)
        smoothed_prob = np.median(current_history)
        
        label = "FAKE" if smoothed_prob > 0.5 else "REAL"
        color = (0, 0, 255) if label == "FAKE" else (0, 255, 0)
        
        text = f"{label}: {smoothed_prob:.2f} (Raw: {prob_ensemble:.2f})"

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    
    active_trackers = current_frame_trackers

    end_time = time.perf_counter()
    processing_time = end_time - start_time
    fps = 1 / processing_time if processing_time > 0 else 0

    stats_text = f"Total processing time: {processing_time * 1000:.1f} ms | FPS: {fps:.1f} | Faces: {len(faces)}"
    cv2.putText(frame, stats_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow('Deepfake Detector - POC', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773902915.946388    9045 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773902916.126407    9060 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Compiling
Compiling done


QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot fi

KeyboardInterrupt: 